# habituation-vad: feature extraction on Kaggle (W1.2 + G1)

**Before running**
1. Settings → Accelerator → **GPU T4** (T4 x2 is fine; one GPU is used). Avoid P100: newer PyTorch builds may not support it.
2. Settings → **Internet on** (needs a phone-verified Kaggle account), for `pip` and the CLIP weights.
3. Add Input → your private dataset holding the `habituation-vad` repository (zip of the folder).
4. Add Input → **ShanghaiTech Campus**, **both parts** (the video anomaly dataset, not the MCNN crowd-counting one). Either one dataset with train + test, or the two Kaggle parts `nikanvasei/shanghaitech-campus-dataset` (train) and `nikanvasei/shanghaitech-campus-dataset-test` (test). The test part must include the frame labels (`01_0014.npy` …).

Then **Run All** (or *Save Version → Save & Run All* for a background run of up to 12 h).
One pass extracts rgb + motion features for 1×1, 2×2 and 3×3 regions, all 13 scenes. Outputs land in `/kaggle/working/`; use *Save Version* (Save & Run All) to keep them,
or download `features_*.zip` from the Output panel.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install open_clip_torch==2.24.0
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "NO GPU. Right panel -> Session options -> Accelerator -> GPU T4 x2, then Run All again. "
    "(On CPU each video takes minutes instead of seconds.)")

In [ ]:
# copy the repository from /kaggle/input to a writable place
import glob, os, shutil, sys
hits = glob.glob('/kaggle/input/**/src/preprocess/extract.py', recursive=True)
assert hits, 'repository not found: add the habituation-vad dataset as an input'
src_repo = hits[0].rsplit('/src/preprocess/', 1)[0]
REPO = '/kaggle/working/habituation-vad'
if not os.path.exists(REPO):
    shutil.copytree(src_repo, REPO)
os.chdir(REPO); sys.path.insert(0, REPO)
print('repo:', REPO)

In [ ]:
# Collect ShanghaiTech Campus from ALL inputs and sort videos by their OFFICIAL names:
#   official training videos are named 01_001 (3 digits)  -> normal only, go into memory
#   official test videos are named     01_0014 (4 digits) -> evaluated with frame labels
# This matters because some Kaggle copies use a re-split that puts official test
# videos (with anomalies) into a "train" folder. Each video is linked into
#   /kaggle/working/sht/training/..., testing/..., testing/test_frame_mask/
import re, shutil, numpy as np
TRAIN_RE = re.compile(r'^\d{2}_\d{3}$')
TEST_RE = re.compile(r'^\d{2}_\d{4}$')
IMG, VID = ('.jpg', '.jpeg', '.png', '.bmp'), ('.avi', '.mp4')
EXPECTED = {'train': 330, 'test': 107}          # official ShanghaiTech Campus totals

def is_frame_dir(p):
    for e in os.scandir(p):
        return e.name.lower().endswith(IMG)
    return False

def better(new, old):      # prefer a path that says "frames" (not optical flow etc.)
    return 'frame' in new.lower() and 'frame' not in old.lower()

src = {'train': {}, 'test': {}}
labels = {}
for d, dirnames, files in os.walk('/kaggle/input', followlinks=True):
    keep = []
    for n in dirnames:
        split = 'train' if TRAIN_RE.match(n) else 'test' if TEST_RE.match(n) else None
        if split is None:
            keep.append(n); continue
        p = os.path.join(d, n)
        if is_frame_dir(p) and (n not in src[split] or better(p, src[split][n][1])):
            src[split][n] = ('frames', p)
    dirnames[:] = keep                          # never walk inside a video's frames
    for f in files:
        stem, ext = os.path.splitext(f)
        split = 'train' if TRAIN_RE.match(stem) else 'test' if TEST_RE.match(stem) else None
        if split and ext.lower() in VID:
            src[split].setdefault(stem, ('videos', os.path.join(d, f)))
        elif split == 'test' and ext == '.npy' and stem not in labels:
            if np.load(os.path.join(d, f), mmap_mode='r').ndim == 1:   # frame labels, not pixel masks
                labels[stem] = os.path.join(d, f)

for split in ('train', 'test'):
    kinds = {k for k, _ in src[split].values()}
    print(f'{split}: {len(src[split])} videos found (official: {EXPECTED[split]}), stored as {kinds or "-"}')
print(f'frame labels found for {sum(v in labels for v in src["test"])} / {len(src["test"])} test videos')

missing_lab = sorted(v for v in src['test'] if v not in labels)
assert len(src['train']) == EXPECTED['train'] and len(src['test']) == EXPECTED['test'], (
    'Incomplete dataset. Add BOTH Kaggle parts as inputs: '
    'nikanvasei/shanghaitech-campus-dataset AND nikanvasei/shanghaitech-campus-dataset-test')
assert not missing_lab, f'test videos without frame labels: {missing_lab[:10]}'
for split in ('train', 'test'):
    assert len({k for k, _ in src[split].values()}) == 1, f'{split}: mix of frame folders and video files'

DATA = '/kaggle/working/sht'
shutil.rmtree(DATA, ignore_errors=True)
for split, top in (('train', 'training'), ('test', 'testing')):
    for name, (kind, p) in src[split].items():
        link = f'{DATA}/{top}/{kind}/{os.path.basename(p)}'
        os.makedirs(os.path.dirname(link), exist_ok=True)
        os.symlink(p, link)
os.makedirs(f'{DATA}/testing/test_frame_mask')
for name in src['test']:
    os.symlink(labels[name], f'{DATA}/testing/test_frame_mask/{name}.npy')
# frame file naming: text sort and numeric sort must agree (the code sorts numerically)
kind, p = next(iter(src['test'].values()))
if kind == 'frames':
    names = [e.name for e in os.scandir(p)]
    num = sorted(names, key=lambda n: [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', n)])
    print('frame names look like:', num[:3], '...', num[-1],
          '| zero-padded' if sorted(names) == num else '| NOT zero-padded (handled by numeric sort)')
print('data_root:', DATA)

In [ ]:
# write a Kaggle config from the shared one: same settings, Kaggle paths
import yaml
cfg = yaml.safe_load(open('configs/shanghaitech.yaml'))
cfg['data_root'] = DATA
cfg['features_root'] = '/kaggle/working/features'
SCENES = None                      # None = all 13 scenes; or e.g. ['04', '05']
cfg['scenes'] = SCENES
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'), sort_keys=False)
print(open('configs/kaggle.yaml').read())
print('outputs per video:', cfg['extract'])

In [ ]:
# 1-minute smoke test: 2 test videos
!python -m src.preprocess.extract --config configs/kaggle.yaml --split test --limit 2

In [ ]:
# full extraction: ONE pass per video produces every output in `extract:`
#   rgb and diff (motion) features, for 1x1, 2x2 and 3x3 regions (6 feature folders).
# Restartable: outputs already written for a video are skipped.
# For a long run, use Save Version -> "Save & Run All (Commit)": it runs in the
# background for up to 12 h, so the browser can be closed.
!python -m src.preprocess.extract --config configs/kaggle.yaml

In [ ]:
!python -m scripts.check_counts --config configs/kaggle.yaml
# baseline AUC for every feature type (CPU, a few minutes)
import copy
from scripts.baseline import run as baseline
base = yaml.safe_load(open('configs/kaggle.yaml')); base['_config_path'] = 'configs/kaggle.yaml'
for chans, grid in [(['rgb'], 2), (['diff'], 2), (['rgb', 'diff'], 2), (['rgb'], 1), (['rgb'], 3)]:
    c = copy.deepcopy(base)
    c['features'] = {'channels': chans}
    print(f'\n===== channels {chans}, grid {grid}x{grid}')
    baseline(c, grid=grid)

In [ ]:
# package the outputs for download (put the zip in your capstone folder)
import shutil
shutil.make_archive('/kaggle/working/features_shanghaitech_all', 'zip', '/kaggle/working/features')
for f in ['results/runs.csv'] + glob.glob('results/baseline_*.csv'):
    shutil.copy(f, '/kaggle/working/')
print(os.listdir('/kaggle/working'))